# Lagrangiano Aumentado

In [76]:
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

In [67]:
def loss_LA(x, f, h, g, lam, rho, mu):
    hx = h(x)
    gx = g(x)

    igualdade = np.dot(lam, hx) + (rho/ 2.0) * np.dot(hx, hx)
    
    phi = np.maximum(0.0, mu+ rho*gx)
    desigualdade = (1.0/ (2.0 * rho)) * (np.dot(phi, phi) - np.dot(mu, mu))
    return f(x) + igualdade + desigualdade

In [125]:
def solver(f, h, g, rho0, gamma, eta, max_iter, tol_via, max_inner = 500, x0 = np.ones(2), otimizador = None):
    x = x0.copy()
    m = len(h(x))
    p = len(g(x))
    lam = np.zeros(m)
    mu = np.zeros(p)
    rho = rho0
    viol_ant = np.inf
    for k in range(max_iter):
        #Passo 1: Minimizar a Loss
        def objetivo(x_):
            return loss_LA(x_, f, h, g, lam, rho, mu)
        if otimizador == None:
            resultado = minimize(
                objetivo,
                x,
                method="L-BFGS-B",
                options={"maxiter": max_inner, "f_tol": 1e-12, "gtol": 1e-8}
            )
        else:
            resultado = otimizador(objetivo, x)
        x = resultado.x

        #Passo 2: Avaliar viabilidade
        hx = h(x)
        gx = g(x)

        viol_h = np.linalg.norm(hx, ord=np.inf)
        viol_g = np.linalg.norm(np.maximum(0, gx), ord=np.inf)
        violacao = max(viol_h, viol_g)
        
        #Passo 3: Atualização
        lam = lam + rho * hx
        mu = np.maximum(0.0, mu + rho * gx)

        if violacao > eta * viol_ant:
            rho = gamma * rho
        viol_ant = violacao

        # Passo 5: Convergência
        if viol_h < tol_via and viol_g < tol_via:
            print(f"\n Convergiu em {k+1} iterações")
            break
    return {
        "x": x,
        "f": f(x),
        "lam": lam,
        "mu": mu,
        "rho": rho,
        "convergido": violacao < tol_via
    }


Teste:

In [126]:
def f(x):
    return (1 - x[0])**2 + 100* (x[1] - x[0]**2)**2

In [127]:
def h(x):
    return np.array([x[0]**2 + x[1]**2 - 1.0])

In [128]:
def g(x):
    return np.array([-(x[0] +  x[1])])

In [129]:
solver(f =f, h =h, g = g, rho0=1.0, gamma=2.0, eta=0.25, tol_via=1e-7, max_iter=50)


 Convergiu em 13 iterações


{'x': array([0.7864151 , 0.61769832]),
 'f': np.float64(0.04567481826109033),
 'lam': array([0.12144786]),
 'mu': array([0.]),
 'rho': 64.0,
 'convergido': np.True_}